<!-- Encabezado con dos logos -->
<table style="width:100%; border:0px">
  <tr style="border:0px">
    <td align="left" style="border:0px">
      <img src="https://www.uclm.es/images/logos/Logo_uclm_40.png" alt="Universidad" height="80">
    </td>
    <td align="right" style="border:0px">
      <img src="https://simd.i3a.uclm.es/wp-content/uploads/2025/01/LOGO-Pie-SIMD-1.webp" alt="Grupo de Investigación" height="80">
    </td>
  </tr>
</table>

<br>

<!-- Título del documento -->
<h1 align="center"> Homework 1.- Particle Filters </h1>

<!-- Información institucional -->
<p align="center">
  <strong>Degree in Computer Science</strong>
  <br>  
  Universidad de Castilla-La Mancha  
  <br>
  <strong>Teacher:</strong> Jesús Martínez Gómez (Jesus.Martinez@uclm.es)
</p>

# Presentation

## Monte Carlo Localization

In this notebook you will implement the core steps of **Monte Carlo Localization (MCL)** for a mobile robot moving in a 2D world with a simulated LIDAR.

### What you must implement

Complete the following functions in the **Monte Carlo Localization** section:

- `mcl_initialise`
- `mcl_predict`
- `mcl_update`
- `mcl_resample`
- `mcl_estimate_pose`

### What is already provided

The notebook already includes:

- Global parameters
- Reusable helpers for angles, motion, particles and LIDAR
- World generation and robot simulation
- LIDAR simulation
- Navigation and simulation engine
- Visualisation tools
- Test cases and experiment runners

Please modify **only** the MCL section unless explicitly instructed otherwise.

### Evaluation overview

Your work will be evaluated in three blocks:

1. **Basic MCL implementation** on three trajectories without kidnappings.
2. **Analysis and improvement** of MCL under robot kidnappings.
3. **Experimental study** of the influence of the number of particles.

# Evaluation Details

## Assessment rubric summary

The practice will be graded with the following weights:

- **50%** Basic MCL implementation
- **30%** Analysis and improvement under kidnappings
- **20%** Study of the influence of the number of particles

### Basic MCL implementation

Evaluate your implementation on three base cases:

- `base_simple`: simple trajectory
- `base_limited_turns`: trajectory with limited turns
- `base_many_turns`: trajectory with many turns

### Kidnapped robot analysis

Evaluate the base implementation and a proposed improvement on:

- `kidnap_once`
- `kidnap_twice`
- `kidnap_three_times`

### Particle-count study

Generate plots showing the effect of the number of particles on localisation accuracy across the three base cases.

## Imports and reusable helpers **(do not modify / add new methods if needed)**

In [ ]:
import copy
import math
import pandas as pd
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation

plt.rcParams["animation.embed_limit"] = 80.0


# ---------------------------------------------------------------------------
# Level 1 helpers: angles, poses, particles
# ---------------------------------------------------------------------------
def wrap_pi(angle):
    """Wrap angle(s) to [-pi, pi). Works with scalars and NumPy arrays."""
    return (np.asarray(angle) + np.pi) % (2.0 * np.pi) - np.pi


def angle_diff(a, b):
    """Smallest signed angular difference a - b."""
    return wrap_pi(np.asarray(a) - np.asarray(b))


def apply_motion(x: float, y: float, th: float, motion_local: np.ndarray) -> np.ndarray:
    """Apply a planar motion expressed in the local frame of heading th."""
    dx_local, dy_local, dth = np.asarray(motion_local, dtype=float)
    c = math.cos(th)
    s = math.sin(th)
    nx = x + c * dx_local - s * dy_local
    ny = y + s * dx_local + c * dy_local
    nth = float(wrap_pi(th + dth))
    return np.array([nx, ny, nth], dtype=float)


def normalise_weights(weights: np.ndarray) -> np.ndarray:
    """Return normalised non-negative weights."""
    weights = np.asarray(weights, dtype=float)
    total = float(np.sum(weights))
    if total <= 0.0 or not np.isfinite(total):
        return np.ones_like(weights, dtype=float) / len(weights)
    return weights / total


def effective_n(weights: np.ndarray) -> float:
    """Effective number of particles, N_eff = 1 / sum(w_i^2)."""
    weights = normalise_weights(weights)
    return float(1.0 / np.sum(np.square(weights)))


# ---------------------------------------------------------------------------
# Level 2 helpers: LIDAR
# ---------------------------------------------------------------------------
def lidar_ray_angles(n_rays: int, fov: float) -> np.ndarray:
    return np.linspace(-fov / 2.0, fov / 2.0, int(n_rays), dtype=float)


def lidar_ray_directions(theta: float, ray_angles: np.ndarray) -> np.ndarray:
    return wrap_pi(theta + np.asarray(ray_angles, dtype=float))


def clip_lidar_ranges(ranges: np.ndarray, range_max: float) -> np.ndarray:
    return np.clip(np.asarray(ranges, dtype=float), 0.0, float(range_max))


def lidar_scan_error(z_obs: np.ndarray, z_pred: np.ndarray) -> float:
    diff = np.asarray(z_obs, dtype=float) - np.asarray(z_pred, dtype=float)
    return float(np.mean(np.square(diff)))


# ---------------------------------------------------------------------------
# Random number generator helpers
# ---------------------------------------------------------------------------
def spawn_rngs(master_seed: int) -> dict:
    """Create independent generators derived from a single visible master seed."""
    seed_seq = np.random.SeedSequence(int(master_seed))
    labels = ["world", "sensor", "mcl", "goal"]
    children = seed_seq.spawn(len(labels))
    return {label: np.random.default_rng(child) for label, child in zip(labels, children)}


## Global parameters


In [ ]:
# ---------------------------------------------------------------------------
# Global parameters (single visible configuration zone)
# ---------------------------------------------------------------------------
MASTER_SEED = 7

# Environment
WORLD_H = 80
WORLD_W = 120
CELL_SIZE = 0.05  # metres per cell
SCATTERED_SMALL_BLOCKS = 12

# Robot and simulation
DT = 0.1
V_MAX = 0.15
W_MAX = 1.2
V_LINEAR = 0.15
BODY_RADIUS = 0.10
N_STEPS = 300

# LIDAR
LIDAR_N_RAYS = 40
LIDAR_RANGE_MAX = 8.0
LIDAR_FOV = 2.0 * math.pi
LIDAR_STEP = 0.02
LIDAR_NOISE_STD = 0.01
LIDAR_CMAP_NAME = "coolwarm"
LIDAR_ALPHA = 0.30

# Navigation
NAV_MIN_FRONT_CLEARANCE = 0.20
NAV_CLEARANCE = 0.50
NAV_TURN_GAIN = 1.8

# MCL
MCL_N_PARTICLES = 400
MCL_INIT_STD = np.array([0.08, 0.08, 0.20], dtype=float)
MCL_MOTION_NOISE = np.array([0.02, 0.02, 0.04], dtype=float)
MCL_ROUGHENING_STD = np.array([0.015, 0.015, 0.03], dtype=float)
MCL_SENSOR_RAY_STRIDE = 5
MCL_SENSOR_SIGMA = 0.20
MCL_RESAMPLE_NEFF_RATIO = 0.50
MCL_RANDOM_PARTICLE_RATIO = 0.05


## 2D model of the world (**do not modify**)


In [ ]:
def make_world(h: int, w: int, rng: np.random.Generator) -> np.ndarray:
    grid = np.zeros((h, w), dtype=np.uint8)

    grid[0, :] = 1
    grid[-1, :] = 1
    grid[:, 0] = 1
    grid[:, -1] = 1

    for _ in range(SCATTERED_SMALL_BLOCKS):
        y = int(rng.integers(5, h - 5))
        x = int(rng.integers(5, w - 5))
        grid[y:y + 2, x:x + 2] = 1

    return grid


def world_to_cell(x_m: float, y_m: float):
    c = int(x_m / CELL_SIZE)
    r = int(y_m / CELL_SIZE)
    return r, c


def in_bounds_rc(r: int, c: int, grid: np.ndarray) -> bool:
    return 0 <= r < grid.shape[0] and 0 <= c < grid.shape[1]


def is_occupied_xy(x_m: float, y_m: float, grid: np.ndarray) -> bool:
    r, c = world_to_cell(x_m, y_m)
    if not in_bounds_rc(r, c, grid):
        return True
    return bool(grid[r, c] == 1)


preview_rngs = spawn_rngs(MASTER_SEED)
grid_preview = make_world(WORLD_H, WORLD_W, preview_rngs["world"])

plt.figure(figsize=(7, 4))
plt.imshow(grid_preview, origin="lower", interpolation="nearest")
plt.title("Occupancy grid map")
plt.xlabel("x (cells)")
plt.ylabel("y (cells)")
plt.show()


## Robot simulator (**do not modify**)


In [ ]:
@dataclass
class RobotState:
    x: float
    y: float
    th: float


@dataclass
class SimConfig:
    dt: float = DT
    v_max: float = V_MAX
    w_max: float = W_MAX
    body_radius: float = BODY_RADIUS


def _collides(x: float, y: float, grid: np.ndarray, body_radius: float) -> bool:
    for ang in np.linspace(0.0, 2.0 * math.pi, 8, endpoint=False):
        px = x + body_radius * math.cos(ang)
        py = y + body_radius * math.sin(ang)
        if is_occupied_xy(px, py, grid):
            return True
    return False


def step_sim(state_true: RobotState, v_cmd: float, w_cmd: float, grid: np.ndarray, cfg: SimConfig) -> RobotState:
    v = float(np.clip(v_cmd, -cfg.v_max, cfg.v_max))
    w = float(np.clip(w_cmd, -cfg.w_max, cfg.w_max))

    motion_local = np.array([v * cfg.dt, 0.0, w * cfg.dt], dtype=float)
    nx, ny, nth = apply_motion(state_true.x, state_true.y, state_true.th, motion_local)

    if _collides(float(nx), float(ny), grid, cfg.body_radius):
        nx, ny = state_true.x, state_true.y

    return RobotState(float(nx), float(ny), float(nth))


def relative_odometry(prev_state: RobotState, new_state: RobotState) -> np.ndarray:
    """Odometry increment expressed in the previous robot local frame."""
    dx_w = float(new_state.x - prev_state.x)
    dy_w = float(new_state.y - prev_state.y)
    c = math.cos(prev_state.th)
    s = math.sin(prev_state.th)
    dx_local = c * dx_w + s * dy_w
    dy_local = -s * dx_w + c * dy_w
    dth = float(angle_diff(new_state.th, prev_state.th))
    return np.array([dx_local, dy_local, dth], dtype=float)


def pose_error(true_pose: RobotState, est_pose: Optional[RobotState]) -> Optional[float]:
    if est_pose is None:
        return None
    return float(math.hypot(true_pose.x - est_pose.x, true_pose.y - est_pose.y))


def sample_random_free_pose(grid: np.ndarray, rng: np.random.Generator, margin: float = BODY_RADIUS) -> RobotState:
    h, w = grid.shape
    for _ in range(10000):
        x = float(rng.uniform(margin, w * CELL_SIZE - margin))
        y = float(rng.uniform(margin, h * CELL_SIZE - margin))
        th = float(rng.uniform(-math.pi, math.pi))
        if not _collides(x, y, grid, margin):
            return RobotState(x, y, th)
    raise RuntimeError("Could not sample a free pose.")


## LIDAR simulation (**do not modify**)


In [ ]:
def cast_single_ray(px: float, py: float, abs_ang: float, grid: np.ndarray,
                    range_max: float = LIDAR_RANGE_MAX, step: float = LIDAR_STEP) -> float:
    dist = 0.0
    while dist < range_max:
        x = px + dist * math.cos(abs_ang)
        y = py + dist * math.sin(abs_ang)
        if is_occupied_xy(x, y, grid):
            return float(dist)
        dist += step
    return float(range_max)


def lidar_scan(state_true: RobotState,
               grid: np.ndarray,
               rng: np.random.Generator,
               n_rays: int = LIDAR_N_RAYS,
               fov: float = LIDAR_FOV,
               range_max: float = LIDAR_RANGE_MAX,
               step: float = LIDAR_STEP,
               noise_std: float = LIDAR_NOISE_STD):

    angles = lidar_ray_angles(n_rays, fov)
    abs_angles = lidar_ray_directions(state_true.th, angles)
    dists = np.empty(len(angles), dtype=float)

    for i, abs_ang in enumerate(abs_angles):
        dists[i] = cast_single_ray(state_true.x, state_true.y, float(abs_ang), grid,
                                   range_max=range_max, step=step)

    if noise_std > 0.0:
        dists = dists + rng.normal(0.0, noise_std, size=dists.shape)

    dists = clip_lidar_ranges(dists, range_max)
    return angles, dists


## Monte Carlo Localization (🧩 **opened for modifications**)

Use the helpers provided above to implement the five core steps of MCL.

Recommended flow:

`initialise → predict → update → resample → estimate`

### Notes

- Particles must remain in free space.
- Don't forget to weight particles and normalised weights.
- Use the simulated LIDAR to compare observed and predicted scans.
- Keep the implementation as clear and readable as possible.

In [ ]:
@dataclass
class Particles:
    states: np.ndarray   # shape (N, 3) -> [x, y, th]
    weights: np.ndarray  # shape (N,)


def make_uniform_particles(states: np.ndarray) -> Particles:
    n = len(states)
    return Particles(states=np.asarray(states, dtype=float),
                     weights=np.ones(n, dtype=float) / n)


def mcl_initialise(n: int,
                   grid: np.ndarray,
                   rng: np.random.Generator,
                   init_pose: Optional[tuple] = None,
                   init_std: np.ndarray = MCL_INIT_STD) -> Particles:
    """TODO: initialise a valid set of particles with uniform weights.

    Expected behaviour:
    - if init_pose is None, sample particles over the free space of the map
    - otherwise, sample around init_pose using init_std
    - ensure all particles are collision-free
    """
    #raise NotImplementedError("Complete mcl_initialise")
    states = np.empty((n, 3), dtype=float)

    if init_pose is None:
        # sample uniformly over the free space of the map
        for i in range(n):
            s = sample_random_free_pose(grid, rng)
            states[i] = [s.x, s.y, s.th]
    else:
        # sample around init_pose using init_std, ensuring collision-free particles
        for i in range(n):
            for _ in range(1000):
                # try sampling a particle around init_pose
                x = float(rng.normal(init_pose[0], init_std[0]))
                y = float(rng.normal(init_pose[1], init_std[1]))
                th = float(rng.normal(init_pose[2], init_std[2]))
                if not _collides(x, y, grid, BODY_RADIUS):
                    states[i] = [x, y, wrap_pi(th)]
                    break
            else:
                # if not possible after 1000 attempts, sample randomly from the free space
                s = sample_random_free_pose(grid, rng)
                states[i] = [s.x, s.y, s.th]
    
    # return particles with uniform weights
    return make_uniform_particles(states)


def mcl_predict(p: Particles,
                odom_delta: np.ndarray,
                motion_noise: np.ndarray,
                grid: np.ndarray,
                rng: np.random.Generator) -> Particles:
    """TODO: propagate each particle using odometry and motion noise."""
    #raise NotImplementedError("Complete mcl_predict")
    states = np.copy(p.states)
    weights = np.copy(p.weights)

    for i in range(len(states)):
        # apply odometry with noise
        delta_noisy = odom_delta + rng.normal(0.0, motion_noise, size=odom_delta.shape)
        states[i] = apply_motion(states[i][0], states[i][1], states[i][2], delta_noisy)

        # ensure collision-free particles
        if _collides(states[i][0], states[i][1], grid, BODY_RADIUS):
            # resample noisy motion until collision-free or fallback to random free pose
            for _ in range(1000):
                delta_noisy = odom_delta + rng.normal(0.0, motion_noise, size=odom_delta.shape)
                new_state = apply_motion(states[i][0], states[i][1], states[i][2], delta_noisy)
                if not _collides(new_state[0], new_state[1], grid, BODY_RADIUS):
                    states[i] = new_state
                    break
            else:
                s = sample_random_free_pose(grid, rng)
                states[i] = [s.x, s.y, s.th]

    return Particles(states=states, weights=weights)


def mcl_update(p: Particles,
               scan_angles: np.ndarray,
               scan_dists: np.ndarray,
               grid: np.ndarray,
               range_max: float = LIDAR_RANGE_MAX,
               ray_stride: int = MCL_SENSOR_RAY_STRIDE,
               sigma: float = MCL_SENSOR_SIGMA) -> Particles:
    """TODO: compute particle weights from the similarity between
    the observed LIDAR scan and the predicted scan of each particle."""
    #raise NotImplementedError("Complete mcl_update")
    weights = np.copy(p.weights)

    for i in range(len(p.states)):
        # predict scan for particle i
        pred_scan = np.empty_like(scan_dists)

        for j in range(0, len(scan_angles), ray_stride):
            # compute absolute angle of ray j for particle i
            abs_ang = wrap_pi(p.states[i][2] + scan_angles[j])
            # compute predicted distance for ray j of particle i
            pred_scan[j] = cast_single_ray(p.states[i][0], p.states[i][1], float(abs_ang), grid,
                                           range_max=range_max, step=LIDAR_STEP)

        # compute weight based on scan similarity
        error = lidar_scan_error(scan_dists[::ray_stride], pred_scan[::ray_stride])
        weights[i] = math.exp(-error / (2.0 * sigma**2))

    return Particles(states=p.states, weights=weights)


def mcl_resample(p: Particles,
                 grid: np.ndarray,
                 rng: np.random.Generator,
                 roughening_std: np.ndarray = MCL_ROUGHENING_STD,
                 random_particle_ratio: float = MCL_RANDOM_PARTICLE_RATIO) -> Particles:
    """TODO: resample particles according to their weights.

    You may optionally keep a percentage of random particles to improve
    recovery after kidnappings.
    """
    #raise NotImplementedError("Complete mcl_resample")
    new_states = np.empty_like(p.states)
    weights_norm = normalise_weights(p.weights)  # normalised weights
    n = len(p.states)
    n_random = int(n * random_particle_ratio)  # number of random particles to add
    n_resample = n - n_random  # number of particles to resample from the current set

    if n_resample > 0:
        # systematic resampling (if n_resample == 0, we skip resampling and just add random particles)
        cumsum_weights = np.cumsum(weights_norm)  # cumulative sum of weights
        positions = (rng.random() + np.arange(n_resample)) / n_resample  # equally spaced positions in [0, 1)
        indices = np.searchsorted(cumsum_weights, positions)  # find indices of particles to resample
        new_states[:n_resample] = p.states[indices].copy()

    # roughening to mitigate sample impoverishment
    for i in range(n_resample):
        new_states[i] += rng.normal(0.0, roughening_std, size=roughening_std.shape)
        new_states[i][2] = wrap_pi(new_states[i][2])
        # ensure collision-free particles
        if _collides(new_states[i][0], new_states[i][1], grid, BODY_RADIUS):
            new_states[i] = p.states[indices[i]]  # revert to previous state if collides
        

    # add random particles
    for i in range(n_random):
        s = sample_random_free_pose(grid, rng)
        new_states[n_resample + i] = [s.x, s.y, s.th]

    return make_uniform_particles(new_states)


def mcl_estimate_pose(p: Particles) -> RobotState:
    """TODO: estimate the robot pose from the weighted particle set."""
    #raise NotImplementedError("Complete mcl_estimate_pose")
    # normalise weights
    weights_norm = normalise_weights(p.weights)

    # compute weighted average of particle states
    x_est = np.sum(weights_norm * p.states[:, 0])
    y_est = np.sum(weights_norm * p.states[:, 1])
    
    # compute weighted average of angles using atan2 of weighted sum of unit vectors
    c = np.sum(weights_norm * np.cos(p.states[:, 2]))
    s = np.sum(weights_norm * np.sin(p.states[:, 2]))
    th_est = math.atan2(s, c)

    return RobotState(float(x_est), float(y_est), float(th_est))



## Navigation  (**do not modify**)


In [ ]:
@dataclass
class NavParams:
    min_front_clearance: float = NAV_MIN_FRONT_CLEARANCE
    clearance: float = NAV_CLEARANCE
    v_linear: float = V_LINEAR
    v_max: float = V_MAX
    w_max: float = W_MAX
    turn_gain: float = NAV_TURN_GAIN


def goal_relative_angle(pose: RobotState, goal_xy):
    gx, gy = goal_xy
    ang_goal_abs = math.atan2(gy - pose.y, gx - pose.x)
    return float(angle_diff(ang_goal_abs, pose.th))


def front_clearance(scan_angles, scan_dists):
    front = np.abs(scan_angles) < math.radians(25)
    return float(np.min(scan_dists[front])) if np.any(front) else float(np.min(scan_dists))


def angular_command_from_heading(selected_heading: float, nav: NavParams):
    return float(np.clip(nav.turn_gain * float(wrap_pi(selected_heading)), -nav.w_max, nav.w_max))


def choose_command(pose_for_control: RobotState, goal_xy, scan_angles, scan_dists, prev_cmd, nav: NavParams):
    goal_heading = goal_relative_angle(pose_for_control, goal_xy)
    selected_heading = goal_heading
    v = nav.v_linear
    w = angular_command_from_heading(selected_heading, nav)

    debug = {
        "mode": "normal",
        "goal_heading": goal_heading,
        "selected_heading": selected_heading,
    }
    return float(v), float(w), debug


def correct_command_for_safety(v: float, w: float, scan_angles, scan_dists, nav: NavParams):
    v = float(np.clip(v, -nav.v_max, nav.v_max))
    w = float(np.clip(w, -nav.w_max, nav.w_max))

    if front_clearance(scan_angles, scan_dists) < nav.min_front_clearance:
        v = 0.0

    return v, w


## Simulation engine  (**do not modify**)

The experiment loop stores everything needed for later analysis:

- true pose
- pose used by the controller
- estimated pose from MCL
- particles and weights
- LIDAR measurements
- issued commands


In [ ]:
@dataclass
class RunConfig:
    steps: int = N_STEPS
    master_seed: int = MASTER_SEED
    n_rays: int = LIDAR_N_RAYS
    range_max: float = LIDAR_RANGE_MAX
    start_xyth: tuple = (0.4, 0.4, 0.0)
    goal_xy: tuple = (4.5, 2.5)
    use_mcl: bool = False
    mcl_init_from_start_pose: bool = True
    mcl_random_particle_ratio: float = MCL_RANDOM_PARTICLE_RATIO

    # Optional teaching extensions
    random_goal_every: Optional[int] = 120
    kidnapping_events: Optional[list] = None


@dataclass
class SimulationState:
    cfg: SimConfig
    nav: NavParams
    grid: np.ndarray
    rngs: dict
    state_true: RobotState
    pose_for_control: RobotState
    goal_xy: tuple
    prev_cmd: tuple
    particles: Optional[Particles] = None
    kidnapping_map: Optional[dict] = None


@dataclass
class StepRecord:
    t: int
    state_true: RobotState
    pose_for_control: RobotState
    est_pose: Optional[RobotState]
    particles: Optional[np.ndarray]
    particle_weights: Optional[np.ndarray]
    scan_angles: np.ndarray
    scan_dists: np.ndarray
    command: tuple
    debug: dict
    min_front: float
    dist_to_goal: float
    goal_xy: tuple


@dataclass
class EpisodeResult:
    policy: str
    grid: np.ndarray
    cell_size: float
    n_rays: int
    range_max: float
    goal_xy: tuple
    config: RunConfig
    steps: list


def sample_free_goal(grid: np.ndarray, rng: np.random.Generator, min_dist_to_border: float = BODY_RADIUS) -> tuple:
    pose = sample_random_free_pose(grid, rng, margin=min_dist_to_border)
    return (pose.x, pose.y)


def normalise_kidnapping_events(kidnapping_events):
    events = {}
    if not kidnapping_events:
        return events

    for item in kidnapping_events:
        if len(item) != 2:
            raise ValueError("Each kidnapping event must be (step_idx, (x, y, th)).")
        step_idx, pose = item
        if len(pose) != 3:
            raise ValueError("Kidnapping pose must contain exactly (x, y, th).")
        events[int(step_idx)] = RobotState(float(pose[0]), float(pose[1]), float(pose[2]))
    return events


def create_simulation_state(rc: RunConfig):
    cfg = SimConfig()
    nav = NavParams()
    rngs = spawn_rngs(rc.master_seed)
    grid = make_world(WORLD_H, WORLD_W, rngs["world"])

    x0, y0, th0 = rc.start_xyth
    state_true = RobotState(x=float(x0), y=float(y0), th=float(th0))
    pose_for_control = RobotState(state_true.x, state_true.y, state_true.th)
    particles = None

    if rc.use_mcl:
        init_pose = rc.start_xyth if rc.mcl_init_from_start_pose else None
        particles = mcl_initialise(
            MCL_N_PARTICLES,
            grid=grid,
            rng=rngs["mcl"],
            init_pose=init_pose,
        )
        pose_for_control = mcl_estimate_pose(particles)

    return SimulationState(
        cfg=cfg,
        nav=nav,
        grid=grid,
        rngs=rngs,
        state_true=state_true,
        pose_for_control=pose_for_control,
        goal_xy=tuple(rc.goal_xy),
        prev_cmd=(0.0, 0.0),
        particles=particles,
        kidnapping_map=normalise_kidnapping_events(rc.kidnapping_events),
    )


def maybe_update_goal(sim: SimulationState, rc: RunConfig, t: int):
    changed = False
    if rc.random_goal_every and t > 0 and t % int(rc.random_goal_every) == 0:
        sim.goal_xy = tuple(sample_free_goal(sim.grid, sim.rngs["goal"]))
        changed = True
    return changed


def maybe_kidnap_robot(sim: SimulationState, t: int):
    if sim.kidnapping_map is None or t not in sim.kidnapping_map:
        return False
    kidnapped_pose = sim.kidnapping_map[t]
    sim.state_true = RobotState(kidnapped_pose.x, kidnapped_pose.y, kidnapped_pose.th)
    return True


def simulation_step(sim: SimulationState, rc: RunConfig, t: int):
    goal_changed = maybe_update_goal(sim, rc, t)
    kidnapped = maybe_kidnap_robot(sim, t)

    scan_angles, scan_dists = lidar_scan(
        sim.state_true,
        sim.grid,
        rng=sim.rngs["sensor"],
        n_rays=rc.n_rays,
        range_max=rc.range_max,
        noise_std=LIDAR_NOISE_STD,
    )

    v, w, debug = choose_command(
        sim.pose_for_control,
        sim.goal_xy,
        scan_angles,
        scan_dists,
        sim.prev_cmd,
        sim.nav,
    )
    v, w = correct_command_for_safety(v, w, scan_angles, scan_dists, sim.nav)
    min_front = front_clearance(scan_angles, scan_dists)

    prev_true_state = RobotState(sim.state_true.x, sim.state_true.y, sim.state_true.th)
    sim.state_true = step_sim(sim.state_true, v, w, sim.grid, sim.cfg)
    odom_delta = relative_odometry(prev_true_state, sim.state_true)
    sim.prev_cmd = (float(v), float(w))

    est_pose = None
    particles_copy = None
    weights_copy = None

    if rc.use_mcl and sim.particles is not None:
        sim.particles = mcl_predict(sim.particles, odom_delta, MCL_MOTION_NOISE, sim.grid, sim.rngs["mcl"])
        sim.particles = mcl_update(sim.particles, scan_angles, scan_dists, sim.grid, range_max=rc.range_max)
        neff = effective_n(sim.particles.weights)

        if neff < MCL_RESAMPLE_NEFF_RATIO * sim.particles.states.shape[0]:
            sim.particles = mcl_resample(
                sim.particles,
                sim.grid,
                rng=sim.rngs["mcl"],
                random_particle_ratio=rc.mcl_random_particle_ratio,
            )

        est_pose = mcl_estimate_pose(sim.particles)
        sim.pose_for_control = RobotState(est_pose.x, est_pose.y, est_pose.th)
        particles_copy = sim.particles.states.copy()
        weights_copy = sim.particles.weights.copy()
        debug["neff"] = float(neff)
        debug["mcl_pos_err"] = pose_error(sim.state_true, est_pose)
    else:
        sim.pose_for_control = RobotState(sim.state_true.x, sim.state_true.y, sim.state_true.th)

    dist_to_goal = float(math.hypot(sim.goal_xy[0] - sim.state_true.x, sim.goal_xy[1] - sim.state_true.y))
    debug = copy.deepcopy(debug)
    debug["goal_changed"] = bool(goal_changed)
    debug["kidnapped"] = bool(kidnapped)
    debug["goal_xy"] = tuple(sim.goal_xy)

    return StepRecord(
        t=t,
        state_true=RobotState(sim.state_true.x, sim.state_true.y, sim.state_true.th),
        pose_for_control=RobotState(sim.pose_for_control.x, sim.pose_for_control.y, sim.pose_for_control.th),
        est_pose=None if est_pose is None else RobotState(est_pose.x, est_pose.y, est_pose.th),
        particles=particles_copy,
        particle_weights=weights_copy,
        scan_angles=np.asarray(scan_angles, dtype=float).copy(),
        scan_dists=np.asarray(scan_dists, dtype=float).copy(),
        command=(float(v), float(w)),
        debug=debug,
        min_front=min_front,
        dist_to_goal=dist_to_goal,
        goal_xy=tuple(sim.goal_xy),
    )


def run_experiment(rc: RunConfig = RunConfig()):
    sim = create_simulation_state(rc)
    steps = []

    for t in range(rc.steps):
        steps.append(simulation_step(sim, rc, t=t))

    final_goal_xy = tuple(steps[-1].goal_xy) if steps else tuple(rc.goal_xy)
    return EpisodeResult(
        policy="navigation",
        grid=sim.grid.copy(),
        cell_size=CELL_SIZE,
        n_rays=rc.n_rays,
        range_max=rc.range_max,
        goal_xy=final_goal_xy,
        config=rc,
        steps=steps,
    )


## Visualisation from stored episodes  (**do not modify**)

These functions do not run the simulator again. They only use the stored `EpisodeResult`.

The allow to:

- simulate once
- inspect many steps afterwards
- compare policies without rerunning visualisation logic

In [ ]:
class WorldRenderer2D:
    def __init__(self, grid: np.ndarray, cell_size: float, goal_xy, n_rays: int, range_max: float,
                 cmap_name: str = "coolwarm", lidar_stride: int = 1, figsize=(11.2, 7.0)):
        self.grid = grid
        self.cell_size = cell_size
        self.goal_xy = goal_xy
        self.lidar_stride = max(1, int(lidar_stride))
        self.range_max = float(range_max)
        self.cmap = plt.get_cmap(cmap_name + "_r")

        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.fig.subplots_adjust(right=0.74)
        self.ax.set_aspect("equal")
        self.ax.set_xlim(0, grid.shape[1] * cell_size)
        self.ax.set_ylim(0, grid.shape[0] * cell_size)
        self.ax.set_facecolor("#f8f9fb")
        self.ax.imshow(
            grid,
            origin="lower",
            extent=[0, grid.shape[1] * cell_size, 0, grid.shape[0] * cell_size],
            alpha=0.28,
            interpolation="nearest",
            cmap="Greys",
        )
        self.ax.set_xlabel("x (m)")
        self.ax.set_ylabel("y (m)")
        self.ax.grid(True, alpha=0.18, linewidth=0.6)

        self.goal_sc = self.ax.scatter([goal_xy[0]], [goal_xy[1]], marker="*", s=250, color="gold",
                                       edgecolors="black", linewidths=0.8, zorder=6, label="current goal")
        self.true_traj_line, = self.ax.plot([], [], linewidth=2.4, color="tab:blue",
                                            label="true trajectory", zorder=4)
        self.est_traj_line, = self.ax.plot([], [], linestyle="--", linewidth=2.0, color="tab:orange",
                                           alpha=0.95, label="estimated trajectory", zorder=4)
        self.rendered_n_rays = max(1, (n_rays + self.lidar_stride - 1) // self.lidar_stride)
        self.lidar_lines = [
            self.ax.plot([], [], linewidth=1.0, alpha=LIDAR_ALPHA, zorder=2)[0]
            for _ in range(self.rendered_n_rays)
        ]

        self.robot_patch = plt.Polygon(np.zeros((3, 2)), closed=True, alpha=0.95, color="tab:blue", zorder=7)
        self.ax.add_patch(self.robot_patch)

        self.est_patch = plt.Polygon(np.zeros((3, 2)), closed=True, alpha=0.45, color="tab:orange",
                                     linestyle="--", zorder=6)
        self.ax.add_patch(self.est_patch)

        self.particle_sc = self.ax.scatter([], [], s=20, c=[[0.12, 0.47, 0.71, 0.60]],
                                   linewidths=0, zorder=5, label="particles")

        self.legend = self.ax.legend(
            loc="upper left",
            bbox_to_anchor=(1.01, 1.0),
            borderaxespad=0.0,
            frameon=True,
            title="Display",
        )
        self.status_text = self.fig.text(
            0.76, 0.58, "", va="top", ha="left", fontsize=9.3, family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="0.8", alpha=0.95),
        )

    def _robot_triangle(self, state: RobotState, length=0.15, width=0.10):
        th = state.th
        p0 = np.array([state.x, state.y])
        p1 = p0 + length * np.array([math.cos(th), math.sin(th)])
        p2 = p0 + width * np.array([math.cos(th + 2.4), math.sin(th + 2.4)])
        p3 = p0 + width * np.array([math.cos(th - 2.4), math.sin(th - 2.4)])
        return np.vstack([p1, p2, p3])

    def _status_summary(self, step: StepRecord, debug: dict):
        flags = []
        if debug.get("goal_changed", False):
            flags.append("goal changed")
        if debug.get("kidnapped", False):
            flags.append("kidnapped")
        flags_text = ", ".join(flags) if flags else "normal"

        if step.est_pose is None:
            est_text = "n/a"
        else:
            est_text = f"({step.est_pose.x:.2f}, {step.est_pose.y:.2f})"

        return "\n".join([
            f"step      : {step.t}",
            f"mode      : {debug.get('mode', 'normal')}",
            f"event     : {flags_text}",
            f"goal      : ({step.goal_xy[0]:.2f}, {step.goal_xy[1]:.2f})",
            f"true pose : ({step.state_true.x:.2f}, {step.state_true.y:.2f})",
            f"est pose  : {est_text}",
            f"dist goal : {step.dist_to_goal:.2f} m",
            f"min front : {step.min_front:.2f} m",
            f"Neff      : {debug.get('neff', float('nan')):.1f}" if "neff" in debug else "Neff      : n/a",
            f"MCL err   : {debug.get('mcl_pos_err', float('nan')):.2f} m" if "mcl_pos_err" in debug else "MCL err   : n/a",
        ])

    def draw(self, step: StepRecord, true_traj_xy, est_traj_xy=None, title=None, show_particles=True):
      debug = step.debug if step.debug is not None else {}

      if title is not None:
          self.ax.set_title(title)

      tx = [p[0] for p in true_traj_xy]
      ty = [p[1] for p in true_traj_xy]
      self.true_traj_line.set_data(tx, ty)

      if est_traj_xy is not None and len(est_traj_xy) > 0:
          ex = [p[0] for p in est_traj_xy]
          ey = [p[1] for p in est_traj_xy]
          self.est_traj_line.set_data(ex, ey)
      else:
          self.est_traj_line.set_data([], [])

      self.goal_sc.set_offsets(np.array([[step.goal_xy[0], step.goal_xy[1]]], dtype=float))

      render_idx = np.arange(0, min(len(step.scan_angles), len(step.scan_dists)), self.lidar_stride)
      for line, i in zip(self.lidar_lines, render_idx):
          x2 = step.state_true.x + step.scan_dists[i] * math.cos(step.state_true.th + step.scan_angles[i])
          y2 = step.state_true.y + step.scan_dists[i] * math.sin(step.state_true.th + step.scan_angles[i])
          line.set_data([step.state_true.x, x2], [step.state_true.y, y2])
          dn = float(np.clip(step.scan_dists[i] / (self.range_max + 1e-9), 0.0, 1.0))
          line.set_color(self.cmap(dn))
          line.set_alpha(LIDAR_ALPHA)

      for line in self.lidar_lines[len(render_idx):]:
          line.set_data([], [])

      self.robot_patch.set_xy(self._robot_triangle(step.state_true))

      if step.est_pose is not None:
          self.est_patch.set_xy(self._robot_triangle(step.est_pose, length=0.13, width=0.08))
          self.est_patch.set_visible(True)
      else:
          self.est_patch.set_visible(False)

      # ---- Partículas: recrear cada frame ----
      if self.particle_sc is not None:
          self.particle_sc.remove()
          self.particle_sc = None

      if self.particle_sc is not None:
        self.particle_sc.remove()

      if show_particles and step.particles is not None and len(step.particles) > 0:
          xy = step.particles[:, :2]
          self.particle_sc = self.ax.scatter(
              xy[:, 0], xy[:, 1],
              s=18,
              c='tab:blue',
              alpha=0.5,
              linewidths=0,
              zorder=5
          )
      else:
          self.particle_sc = self.ax.scatter([], [],s=18,c='tab:blue',alpha=0.5,linewidths=0,
              zorder=5)

      self.status_text.set_text(self._status_summary(step, debug))
      return self.fig


def result_pose_array(result, attr="state_true"):
    rows = []
    for step in result.steps:
        state = getattr(step, attr)
        if state is None:
            rows.append([np.nan, np.nan, np.nan])
        else:
            rows.append([state.x, state.y, state.th])
    return np.array(rows, dtype=float)


def goal_change_indices(result):
    idxs = []
    prev_goal = None
    for i, step in enumerate(result.steps):
        if prev_goal is None or tuple(step.goal_xy) != tuple(prev_goal):
            idxs.append(i)
            prev_goal = tuple(step.goal_xy)
    return idxs


def kidnapping_indices(result):
    return [i for i, step in enumerate(result.steps) if step.debug.get("kidnapped", False)]


def build_step_title(result, step_idx: int):
    step = result.steps[step_idx]
    debug = step.debug
    suffix = []
    if debug.get("goal_changed", False):
        suffix.append("goal changed")
    if debug.get("kidnapped", False):
        suffix.append("kidnapped")
    suffix_text = f" | {' + '.join(suffix)}" if suffix else ""
    return f"step {step.t} | {result.policy} | dist(goal)={step.dist_to_goal:.2f} m{suffix_text}"


def plot_trajectory(result, figsize=(9.0, 6.2), show_estimated=True):
    poses_true = result_pose_array(result, attr="state_true")
    poses_est = result_pose_array(result, attr="est_pose")
    goal_idxs = goal_change_indices(result)
    kidnap_idxs = kidnapping_indices(result)
    goal_xy = np.array([result.steps[i].goal_xy for i in goal_idxs], dtype=float) if goal_idxs else np.empty((0, 2))
    kidnap_xy = np.array([[result.steps[i].state_true.x, result.steps[i].state_true.y] for i in kidnap_idxs], dtype=float) if kidnap_idxs else np.empty((0, 2))

    fig, ax = plt.subplots(figsize=figsize)
    fig.subplots_adjust(right=0.79)
    ax.set_aspect("equal")
    ax.set_title(f"Trajectory | policy={result.policy} | use_mcl={result.config.use_mcl}")
    ax.imshow(
        result.grid,
        origin="lower",
        extent=[0, result.grid.shape[1] * result.cell_size, 0, result.grid.shape[0] * result.cell_size],
        alpha=0.28,
        interpolation="nearest",
        cmap="Greys",
    )
    ax.grid(True, alpha=0.18, linewidth=0.6)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")

    ax.plot(poses_true[:, 0], poses_true[:, 1], linewidth=2.5, color="tab:blue", label="true trajectory", zorder=4)
    ax.scatter([poses_true[0, 0]], [poses_true[0, 1]], marker="o", s=65, color="tab:green", label="start", zorder=6)
    ax.scatter([poses_true[-1, 0]], [poses_true[-1, 1]], marker="X", s=75, color="black", label="end", zorder=6)

    if show_estimated and result.config.use_mcl and not np.all(np.isnan(poses_est)):
        ax.plot(poses_est[:, 0], poses_est[:, 1], linestyle="--", linewidth=2.0, color="tab:orange",
                label="estimated trajectory", zorder=4)

    if len(goal_xy) > 0:
        ax.scatter(goal_xy[:, 0], goal_xy[:, 1], marker="*", s=180, color="gold", edgecolors="black",
                   linewidths=0.8, label="goal changes", zorder=6)
        for k, idx in enumerate(goal_idxs):
            gx, gy = result.steps[idx].goal_xy
            ax.text(gx + 0.03, gy + 0.03, str(k), fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.12", facecolor="white", alpha=0.7, edgecolor="none"))

    if len(kidnap_xy) > 0:
        ax.scatter(kidnap_xy[:, 0], kidnap_xy[:, 1], marker="s", s=60, color="tab:red",
                   label="kidnapping events", zorder=6)

    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0, frameon=True)
    plt.show()


def plot_world_snapshot(result, step_idx: int, lidar_stride_render: int = 2, show_particles: bool = True):
    step_idx = int(np.clip(step_idx, 0, len(result.steps) - 1))
    renderer = WorldRenderer2D(
        result.grid,
        result.cell_size,
        result.steps[step_idx].goal_xy,
        result.n_rays,
        result.range_max,
        cmap_name=LIDAR_CMAP_NAME,
        lidar_stride=lidar_stride_render,
    )
    true_traj_xy = [(s.state_true.x, s.state_true.y) for s in result.steps[:step_idx + 1]]
    est_traj_xy = [(s.est_pose.x, s.est_pose.y) for s in result.steps[:step_idx + 1] if s.est_pose is not None]

    renderer.draw(
        result.steps[step_idx],
        true_traj_xy=true_traj_xy,
        est_traj_xy=est_traj_xy,
        title=build_step_title(result, step_idx),
        show_particles=show_particles,
    )
    plt.show()


def animate_world(result, frame_stride: int = 4, lidar_stride_render: int = 2, show_particles: bool = True):
    renderer = WorldRenderer2D(
        result.grid,
        result.cell_size,
        result.steps[0].goal_xy,
        result.n_rays,
        result.range_max,
        cmap_name=LIDAR_CMAP_NAME,
        lidar_stride=lidar_stride_render,
    )
    frames = list(range(0, len(result.steps), max(1, int(frame_stride))))

    def init():
        renderer.true_traj_line.set_data([], [])
        renderer.est_traj_line.set_data([], [])
        renderer.goal_sc.set_offsets(np.array([[result.steps[0].goal_xy[0], result.steps[0].goal_xy[1]]], dtype=float))
        renderer.robot_patch.set_xy(np.zeros((3, 2)))
        renderer.est_patch.set_xy(np.zeros((3, 2)))
        renderer.est_patch.set_visible(False)
        renderer.particle_sc.remove()
        renderer.particle_sc = renderer.ax.scatter([], [], s=20, c=[[0.12, 0.47, 0.71, 0.60]],
                                                  linewidths=0, zorder=5, label="particles")
        for line in renderer.lidar_lines:
            line.set_data([], [])
        renderer.status_text.set_text("")
        renderer.ax.set_title("")

        return [
            renderer.robot_patch,
            renderer.est_patch,
            renderer.goal_sc,
            renderer.true_traj_line,
            renderer.est_traj_line,
            renderer.particle_sc,
            *renderer.lidar_lines,
        ]

    def update(frame_idx):
        step_idx = frames[frame_idx]
        step = result.steps[step_idx]
        true_traj_xy = [(s.state_true.x, s.state_true.y) for s in result.steps[:step_idx + 1]]
        est_traj_xy = [(s.est_pose.x, s.est_pose.y) for s in result.steps[:step_idx + 1] if s.est_pose is not None]
        renderer.draw(
            step,
            true_traj_xy=true_traj_xy,
            est_traj_xy=est_traj_xy,
            title=build_step_title(result, step_idx),
            show_particles=show_particles,
        )
        return [
            renderer.robot_patch,
            renderer.est_patch,
            renderer.goal_sc,
            renderer.true_traj_line,
            renderer.est_traj_line,
            renderer.particle_sc,
            *renderer.lidar_lines,
        ]

    ani = animation.FuncAnimation(
        renderer.fig,
        update,
        frames=len(frames),
        init_func=init,
        interval=60,
        blit=False,
        repeat=False,
        cache_frame_data=False,
    )
    plt.close(renderer.fig)
    return HTML(ani.to_jshtml(default_mode="once"))

## Test case definitions and experiment runners  (**do not modify**)

The following utilities define the base cases and the kidnapped-robot cases used in the experiments.

Use `run_case(...)` for a single case and `run_case_set(...)` for multiple cases.



In [ ]:
@dataclass(frozen=True)
class TestCase:
    key: str
    name: str
    start_xyth: tuple
    goal_xy: tuple
    steps: int
    seed: int
    kidnapping_events: Optional[list] = None
    description: str = ""


@dataclass
class CaseResult:
    case: TestCase
    episode: EpisodeResult
    mean_position_error: float
    final_position_error: float
    mean_heading_error: float


def compute_position_errors(result: EpisodeResult) -> np.ndarray:
    errors = []
    for step in result.steps:
        if step.est_pose is None:
            continue
        dx = step.state_true.x - step.est_pose.x
        dy = step.state_true.y - step.est_pose.y
        errors.append(math.hypot(dx, dy))
    return np.asarray(errors, dtype=float)


def compute_heading_errors(result: EpisodeResult) -> np.ndarray:
    errors = []
    for step in result.steps:
        if step.est_pose is None:
            continue
        errors.append(abs(angle_diff(step.state_true.th, step.est_pose.th)))
    return np.asarray(errors, dtype=float)


def summarise_case(case: TestCase, episode: EpisodeResult) -> CaseResult:
    pos_err = compute_position_errors(episode)
    th_err = compute_heading_errors(episode)
    return CaseResult(
        case=case,
        episode=episode,
        mean_position_error=float(np.mean(pos_err)) if len(pos_err) else float('nan'),
        final_position_error=float(pos_err[-1]) if len(pos_err) else float('nan'),
        mean_heading_error=float(np.mean(th_err)) if len(th_err) else float('nan'),
    )


def run_case(case: TestCase,
             use_mcl: bool = True,
             mcl_init_from_start_pose: bool = True,
             n_particles: int = MCL_N_PARTICLES,
             random_particle_ratio: float = MCL_RANDOM_PARTICLE_RATIO) -> CaseResult:
    global MCL_N_PARTICLES
    old_n_particles = MCL_N_PARTICLES
    MCL_N_PARTICLES = int(n_particles)
    try:
        episode = run_experiment(
            rc=RunConfig(
                use_mcl=use_mcl,
                steps=case.steps,
                master_seed=case.seed,
                start_xyth=case.start_xyth,
                goal_xy=case.goal_xy,
                mcl_init_from_start_pose=mcl_init_from_start_pose,
                mcl_random_particle_ratio=random_particle_ratio,
                random_goal_every=None,
                kidnapping_events=case.kidnapping_events,
            )
        )
    finally:
        MCL_N_PARTICLES = old_n_particles
    return summarise_case(case, episode)


def run_case_set(cases,
                 use_mcl: bool = True,
                 mcl_init_from_start_pose: bool = True,
                 n_particles: int = MCL_N_PARTICLES,
                 random_particle_ratio: float = MCL_RANDOM_PARTICLE_RATIO):
    return [
        run_case(
            case,
            use_mcl=use_mcl,
            mcl_init_from_start_pose=mcl_init_from_start_pose,
            n_particles=n_particles,
            random_particle_ratio=random_particle_ratio,
        )
        for case in cases
    ]


def case_results_table(results):
    rows = []
    for item in results:
        rows.append({
            'case': item.case.key,
            'mean_position_error': item.mean_position_error,
            'final_position_error': item.final_position_error,
            'mean_heading_error': item.mean_heading_error,
        })
    return rows


BASE_CASES = [
    TestCase(
        key='base_simple',
        name='Base case 1 — simple trajectory',
        start_xyth=(0.4, 0.4, 0.0),
        goal_xy=(4.8, 0.6),
        steps=220,
        seed=11,
        description='Simple trajectory with few turns.',
    ),
    TestCase(
        key='base_limited_turns',
        name='Base case 2 — limited turns',
        start_xyth=(0.4, 0.4, 0.0),
        goal_xy=(4.8, 2.8),
        steps=260,
        seed=21,
        description='Trajectory with moderate heading changes.',
    ),
    TestCase(
        key='base_many_turns',
        name='Base case 3 — many turns',
        start_xyth=(0.4, 0.4, 0.0),
        goal_xy=(5.4, 3.2),
        steps=320,
        seed=31,
        description='Longer trajectory with many turns.',
    ),
]

KIDNAPPING_CASES = [
    TestCase(
        key='kidnap_once',
        name='Kidnapping case 1 — one kidnapping',
        start_xyth=(0.4, 0.4, 0.0),
        goal_xy=(4.8, 2.8),
        steps=260,
        seed=41,
        kidnapping_events=[(110, (3.6, 0.8, 1.2))],
        description='One kidnapping event during the run.',
    ),
    TestCase(
        key='kidnap_twice',
        name='Kidnapping case 2 — two kidnappings',
        start_xyth=(0.4, 0.4, 0.0),
        goal_xy=(5.0, 3.0),
        steps=300,
        seed=51,
        kidnapping_events=[(90, (3.6, 0.8, 1.2)), (190, (1.0, 3.0, -1.0))],
        description='Two kidnapping events during the run.',
    ),
    TestCase(
        key='kidnap_three_times',
        name='Kidnapping case 3 — three kidnappings',
        start_xyth=(0.4, 0.4, 0.0),
        goal_xy=(5.4, 3.2),
        steps=340,
        seed=61,
        kidnapping_events=[
            (80, (3.6, 0.8, 1.2)),
            (170, (1.0, 3.0, -1.0)),
            (260, (5.2, 1.6, 2.4)),
        ],
        description='Three kidnapping events during the run.',
    ),
]



## Experiment 1 — Basic MCL (3 trajectories)

Run the 3 base cases without kidnappings and report:

- a trajectory plot for each case
- the mean and final localisation error
- a short discussion of the behaviour of your implementation (using your own words)

---

In this section, the MCL implementation will be analysed by executing it with three different test cases. The first one is a simple trajectory with a few robot turns. The second one requires additional heading changes. Lastly, the third one is a longer trajectory with more frequent robot turns.

In the following cell, the test cases are simulated and their results are saved to validate the algorithm.

In [ ]:
# TODO: run your implementation on the three base cases
# Example:
base_results = run_case_set(BASE_CASES, use_mcl=True, mcl_init_from_start_pose=False)

Now, the test results are displayed to see if the implementation performs as expected. Three kinds of errors are shown: firstly, the **mean position error** refers to the mean error between the actual robot location and its estimate during the entire simulation. The **final position error** is the last error computed. Finally, the **mean heading error** is the difference between the actual robot heading and its estimate during the simulation.

In [ ]:
# Display results
display(pd.DataFrame(case_results_table(base_results)))

As shown, the *mean position error* is relatively low. Its values range from 0.2 to 0.24 meters, depending on the case. The same applies to the *mean heading error*, whose values range from 0.08 to 0.13. The *final position error* indicates that, ultimately, the robot estimates its location with high precision, as the difference between its actual location and its estimate is just a few centimeters.

An interactive simulation is displayed for each case. In the first one, the starting **MCL error** is very high, at around 4 meters between the actual location and the estimate. Once the robot localises itself on the map, the error decreases to a range of 0.05 to 0.15 meters, proving that the estimation is highly precise. The **number of effective particles** ($N_{eff}$) also starts with an initially low value and, after the robot localises itself, it increases to a range of approximately 150 to 275. A lower $N_{eff}$ value means that there is a particle degradation problem and the filter is wasting computational resources inefficiently.

In [ ]:
# Visualise a representative base case
animate_world(base_results[0].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

In the second simulation, after step 6, the estimate and the actual location converge, increasing the $N_{eff}$ values above 200, and reducing the *MCL error* to approximately 0.05 meters.

In [ ]:
# Visualise a representative base case
animate_world(base_results[1].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

Finally, in the third simulation, there is a similar situation as in the second one. After a few steps, the robot quickly localises itself on the map, increasing the number of effective particles and decreasing the *MCL error*.

In [ ]:
# Visualise a representative base case
animate_world(base_results[2].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

In conclusion, the implementation functions as expected. It takes only a few steps for the robot to localise itself on the map. Once this is achieved, the localization error is quite low; only a few centimeters from the actual robot location. Furthermore, the number of effective particles increases to high values, proving that most of the particles are significantly contributing to the estimation.

## Experiment 2 — Kidnapped robot

1. Run the 3 kidnapping cases with the base implementation.
2. Analyse what happens after each kidnapping event.
3. Propose and implement one improvement to help recovery.
4. Compare the improved behaviour with the base implementation.

A valid improvement is, for example, keeping a percentage of particles random after resampling.

---

In this section, the base implementation will be evaluated using three different test cases in which the robot is kidnapped. Specifically, these cases involve kidnapping the robot once, twice and three times respectively. Subsequently, an already-implemented improvement will be proposed, and the cases will be re-executed to observe if the robot recovers after a kidnapping.

As stated above, the improvement is currently integrated. The MCL implementation adds a random number of particles during the resample step based on the *random_particle_ratio* parameter. For instance, if there are 100 particles and a random particle ratio of 0.05 is specified, 5 of the new particles will be located randomly on the map after the resample step.

In the following cell, the kidnapping cases are executed with a specified *random_particle_ratio* of 0.0; therefore, the improvement does not have any significant effect.

In [ ]:
kidnap_base = run_case_set(KIDNAPPING_CASES, use_mcl=True, mcl_init_from_start_pose=False, random_particle_ratio=0.0)

As shown, the position and heading errors in the three cases are higher than in the previous experiment. This may be caused by the robot having trouble localising itself on the map after the kidnapping. Nevertheless, as previously mentioned, these errors are computed based on the entire simulation; therefore, while the robot may eventually localise correctly, until that happens, it is positioned farther from its actual location during several steps.

In [ ]:
display(pd.DataFrame(case_results_table(kidnap_base)))

The simulation for the first case is displayed in the following cell. As illustrated, in this case, before the kidnapping occurs, the robot's estimate is close to its actual location. However, once the kidnapping occurs, the robot fails to localise itself correctly.

In [ ]:
# one kidnapping event
animate_world(kidnap_base[0].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

Looking at the second simulation, where two kidnapping events occur, it can be seen that the robot does not manage to localise itself correctly after either of them. Since all the particles are concentrated around its estimate and none of them are distributed accross the map, MCL cannot function correctly and estimate the new location.

In [ ]:
# two kidnapping events
animate_world(kidnap_base[1].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

The same occurs in the third case, as shown in the next simulation where three kidnapping events occur. The robot does not manage to localise itself after any of them. In fact, its estimate stays close to the goal throughout the entire simulation.

In [ ]:
# three kidnapping events
animate_world(kidnap_base[2].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

As observed, the robot did not manage to localise itself in any simulation. Since the experiment was conducted without random particles, there was no possibility of one of them being close to the robot's actual location and helping it to recover after a kidnapping event. In general, the number of effective particles was extremely low (certainly lower than in the previous experiment) and the MCL error was fairly high, which was expected as the robot's estimate was located farther from its actual position. These results highlight the need for an improvement in the algorithm to address this problem and assist the robot in localising itself.

In this next step, the kidnapping cases will be executed with a random particle ratio of 0.05 to verify whether having some random particles spread across the map assists the robot in localising itself after the kidnapping events.

In [ ]:
kidnap_improved = run_case_set(KIDNAPPING_CASES, use_mcl=True, mcl_init_from_start_pose=False, random_particle_ratio=0.05)

In general, the results appear to be better than those without random particles. However, the error values remain relatively high. In the first and third cases, the robot does not appear to localise itself correctly. The mean and final position errors are significant, particularly the final position error in the first case. The same applies to the mean heading error in both instances. Nonetheless, the second case, which involved two kidnapping events, concluded with positive results, demonstrating that the robot probably managed to localise itself successfully.

In [ ]:
display(pd.DataFrame(case_results_table(kidnap_improved)))

As foreseen, the robot does not manage to localise itself correctly on the map during the first simulation once it is kidnapped. Its estimate converges towards the goal while the robot continues moving toward the starting position, suggesting it believes it is reaching the target. Since the map is somewhat symmetrical when divided into top and bottom halves, this might be the reason for the estimate pointing to the goal while the robot returns to the start. Therefore, this likely explains the poor results observed in the first case.

In [ ]:
# one kidnapping event (improved strategy)
animate_world(kidnap_improved[0].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

In the second simulation, after the first kidnapping event, the robot estimates its position close to its actual location, though not precisely. Since there are three similar objects in close proximity, the robot is positioned between two of them, but estimates its location at the third one. However, after the second kidnapping, the robot quickly re-localises itself. In general, the robot does not have much trouble determining its real position, which explains the good results observed in this case.

In [ ]:
# two kidnapping events (improved strategy)
animate_world(kidnap_improved[1].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

Finally, in the third simulation where three kidnapping events occur, the robot manages to localise itself in two of them. Initially, it estimates its location near the goal, but after the first kidnapping, it quickly converges to the actual location. A similar pattern is observed after the second kidnapping, where the robot successfully re-localises. However, after the third event, its estimate fails to converge; instead, it moves rapidly across different points of the map without reaching the real location.

In [ ]:
# three kidnapping events (improved strategy)
animate_world(kidnap_improved[2].episode, frame_stride=2, lidar_stride_render=2, show_particles=True)

Overall, the results are inconsistent. Occasionally, the robot does not manage to localise itself despite the use of random particles. While the improvement is valid in some cases, there is room for further additions which might help achieve localisation, such as not depending solely on LiDAR sensors. Tailoring the random particles ratio attending to a specific map might also help. Nonetheless, the use of random particles has improved the results and has assisted the robot compared to not using them.

## Experiment 3 — Influence of the number of particles

Using the three base cases, analyse how the number of particles affects localisation accuracy.

Minimum requirement:

- run the three base cases with several values of `N_particles`
- produce at least one plot of `number of particles` versus `mean localisation error`
- discuss the trade-off between accuracy and computational cost



In [ ]:
PARTICLE_COUNTS = [50, 100, 200, 400, 800]

# TODO: complete the experimental study
# Suggested structure:
# study_rows = []
# for n_particles in PARTICLE_COUNTS:
#     results = run_case_set(BASE_CASES, use_mcl=True, mcl_init_from_start_pose=False, n_particles=n_particles)
#     for item in results:
#         study_rows.append({
#             'case': item.case.key,
#             'n_particles': n_particles,
#             'mean_position_error': item.mean_position_error,
#         })



## Submission checklist

Before submitting, make sure that:

- The five MCL functions are fully implemented
- The three basic cases have been executed and analysed
- The three kidnapped-robot cases have been analysed
- At least one improvement has been implemented and discussed
- The particle-count study includes the requested plots and conclusions
- The notebook runs from top to bottom without manual intervention

In [ ]:
# Add as many cells as needed